# 01 — SMLP4Rec plain (history only), trained on the train-window warm rows

Date split (`configs/split.yaml`), rows from `scripts/expedia_build_split.py`. Training rows: train window, L ≥ 1 (the plain model needs a history). Early stopping on **autoregressive valid NDCG@10** (`configs/train.yaml`). Dropout grid on the reference seed (one-SE rule by user, ties to the higher dropout), then the chosen value for every seed. Every model is scored on valid and test under both protocols: autoregressive (primary) and teacher-forced (secondary). Cold rows (L = 0) get an empty history: one constant ranking, out of the plain model's training distribution (stated in every table).

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / "results" / ("week6_retrain_fast" if FAST else "week6_retrain")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 12


In [2]:
def summary_tables(ranks: dict) -> dict:
    """Metrics per window x protocol x row set (unrounded)."""
    out = {}
    for w in ("valid", "test"):
        masks = row_set_masks(L[w])
        for proto in ("ar", "tf"):
            out[f"{w}_{proto}"] = {rs: metrics(ranks[f"{w}_{proto}"], m) for rs, m in masks.items()}
    return out


def save_ranks(name: str, ranks: dict) -> Path:
    RANKS.mkdir(parents=True, exist_ok=True)
    path = RANKS / f"ranks_{name}.npz"
    np.savez_compressed(path, **{k: v for k, v in ranks.items()}, **{f"event_{w}": rows["event"][i] for w, i in idx.items()})
    return path

In [3]:
from nbp.hybrid.fusion import one_se_pick
from nbp.models.train import TrainConfig, fit, git_hash, run_dir_for

if FAST and TRAIN_CFG["min_delta"] is None:  # smoke run: use the smoke calibration of notebook 00
    TRAIN_CFG["min_delta"] = json.loads((RES / "min_delta_calibration.json").read_text(encoding="utf-8"))["min_delta"]
assert TRAIN_CFG["min_delta"] is not None, "run notebook 00 first (min_delta calibration)"


def train_cfg(seed: int) -> TrainConfig:
    return TrainConfig.from_dict(TRAIN_CFG, seed)


def per_row(r, k=K_MON):
    return (r <= k) / np.log2(r + 1)


def pick_dropout(valid_ranks: dict) -> tuple:
    """One-standard-error rule (user-clustered) on AR valid NDCG@k; ties go to the stronger
    regularisation (higher dropout)."""
    scores = {d: per_row(r) for d, r in valid_ranks.items()}
    return one_se_pick(scores, USERS["valid"], complexity=lambda d: -d)


def run_meta(run_dir: Path, res, extra: dict) -> dict:
    meta = {"run_dir": str(run_dir), "git_commit": git_hash(ROOT), "best_epoch": res.best_epoch,
            "best_monitor": res.best_monitor, "stop_reason": res.stop_reason, "per_epoch": res.per_epoch,
            "train_config": TRAIN_CFG, "model_config": MODEL_CFG, "split": SPLIT, **extra}
    write_json(run_dir / "metrics.json", meta)
    return meta

In [4]:
from nbp.models import scorers as S

train_rows = np.flatnonzero((rows["window"] == TRAIN) & (rows["L"] >= 1))
loss = S.plain_loss(rows["seq"][train_rows], rows["seq_len"][train_rows], rows["y"][train_rows])
ev = R.valid_eval_fn(S.plain_scorer, views["valid"], Y["valid"], probe, y_probe, K_MON, BATCH)
print("training rows (train window, L >= 1):", len(train_rows))


def train_one(seed: int, drop: float):
    cfg = S.model_config(MODEL_CFG, {"seed": seed, "hidden_dropout_prob": drop})
    model = S.build_plain(cfg, N_ITEMS)
    run_dir = run_dir_for(EXP, "expedia", "smlp4rec-plain", f"d{drop}-s{seed}", TODAY)
    res = fit(model, len(train_rows), loss, ev, train_cfg(seed), run_dir)
    return model, res, run_dir

training rows (train window, L >= 1): 1515761


## Dropout grid on the reference seed

In [5]:
grid = {}
for drop in TRAIN_CFG["dropout_grid"]:
    model, res, run_dir = train_one(REF_SEED, drop)
    ranks = R.evaluate(S.plain_scorer(model), {"valid": views["valid"]}, Y, BATCH)
    grid[drop] = {"model": model, "res": res, "run_dir": run_dir, "valid_ar": ranks["valid_ar"]}
    print(drop, "best epoch", res.best_epoch, res.stop_reason, "| AR valid ndcg@10", round(per_row(ranks["valid_ar"]).mean(), 4))
DROP, best_d, eligible = pick_dropout({d: g["valid_ar"] for d, g in grid.items()})
print("chosen dropout", DROP, "| best", best_d, "| within one SE", eligible)

epoch 1: train loss 3.9536 (251.4s) | valid monitor 0.1765 nll 4.3127 (40.4s) | best 1, no gain 0/2


epoch 2: train loss 3.9196 (253.2s) | valid monitor 0.1779 nll 4.2943 (40.1s) | best 2, no gain 0/2


epoch 3: train loss 3.9160 (253.2s) | valid monitor 0.1785 nll 4.2830 (41.9s) | best 2, no gain 1/2


epoch 4: train loss 3.9137 (282.0s) | valid monitor 0.1786 nll 4.3015 (23.5s) | best 2, no gain 2/2
stop: patience; best epoch 2 (monitor 0.1779)


0.0 best epoch 2 patience | AR valid ndcg@10 0.1779


epoch 1: train loss 3.9819 (746.2s) | valid monitor 0.1675 nll 4.3385 (23.0s) | best 1, no gain 0/2


epoch 2: train loss 3.9490 (675.8s) | valid monitor 0.1724 nll 4.3062 (23.1s) | best 2, no gain 0/2


epoch 3: train loss 3.9447 (677.7s) | valid monitor 0.1713 nll 4.3059 (23.2s) | best 2, no gain 1/2


epoch 4: train loss 3.9420 (680.3s) | valid monitor 0.1725 nll 4.3088 (23.1s) | best 2, no gain 2/2
stop: patience; best epoch 2 (monitor 0.1724)


0.2 best epoch 2 patience | AR valid ndcg@10 0.1724
chosen dropout 0.0 | best 0.0 | within one SE [0.0]


## All seeds with the chosen dropout; scores on valid and test under both protocols

In [6]:
results = {}
for seed in SEEDS:
    if seed == REF_SEED:
        model, res, run_dir = grid[DROP]["model"], grid[DROP]["res"], grid[DROP]["run_dir"]
    else:
        model, res, run_dir = train_one(seed, DROP)
    ranks = R.evaluate(S.plain_scorer(model), views, Y, BATCH)
    save_ranks(f"plain_s{seed}", ranks)
    tables = summary_tables(ranks)
    meta = run_meta(run_dir, res, {"model": "SMLP4Rec plain", "seed": seed, "hidden_dropout_prob": DROP,
                                   "training_rows": "train window, L >= 1", "n_training_rows": int(len(train_rows)),
                                   "cold_rows": "empty history, one constant ranking (out of distribution)",
                                   "results": tables})
    results[seed] = meta
    write_json(RES / f"plain_s{seed}.json", {**meta, "dropout_grid": {str(d): {"best_epoch": g["res"].best_epoch, "stop_reason": g["res"].stop_reason, "valid_ar_ndcg@10": float(per_row(g["valid_ar"]).mean())} for d, g in grid.items()}, "dropout_chosen": DROP})
    t = tables["test_ar"]
    print(f"seed {seed}: best epoch {res.best_epoch} ({res.stop_reason}) | test AR all ndcg@10 {t['all']['ndcg@10']:.4f} warm {t['warm']['ndcg@10']:.4f} cold {t['cold']['ndcg@10']:.4f}")

seed 2022: best epoch 2 (patience) | test AR all ndcg@10 0.1843 warm 0.2023 cold 0.1069


epoch 1: train loss 3.9527 (125.4s) | valid monitor 0.1765 nll 4.2823 (23.5s) | best 1, no gain 0/2


epoch 2: train loss 3.9192 (132.0s) | valid monitor 0.1788 nll 4.2733 (23.8s) | best 2, no gain 0/2


epoch 3: train loss 3.9159 (137.8s) | valid monitor 0.1778 nll 4.2951 (24.5s) | best 2, no gain 1/2


epoch 4: train loss 3.9140 (133.3s) | valid monitor 0.1775 nll 4.2992 (23.7s) | best 2, no gain 2/2
stop: patience; best epoch 2 (monitor 0.1788)


seed 2023: best epoch 2 (patience) | test AR all ndcg@10 0.1849 warm 0.2026 cold 0.1091


epoch 1: train loss 3.9559 (124.8s) | valid monitor 0.1774 nll 4.2837 (23.8s) | best 1, no gain 0/2


epoch 2: train loss 3.9207 (143.2s) | valid monitor 0.1780 nll 4.2898 (23.1s) | best 1, no gain 1/2


epoch 3: train loss 3.9169 (124.8s) | valid monitor 0.1784 nll 4.3005 (23.5s) | best 3, no gain 0/2
stop: nll_guard; best epoch 3 (monitor 0.1784)


seed 2024: best epoch 3 (nll_guard) | test AR all ndcg@10 0.1840 warm 0.2014 cold 0.1093
